## Intro

This notebook prepares the `locations_custom_from_metadata.yaml` file for input to the Calliope model. This takes locations produced in the maxpregions notebook as well as region data and prepares it to be fed into the calliope model.

Areas must be delivered in km2 and power in MW. 
Costs at ouput are now given in k$.

In [4]:
import pandas as pd
import yaml
from itertools import combinations
from tqdm import tqdm  # Import tqdm for progress bars

cluster_method = 'MaxP_Rook'  # Agglomerative, Spectral, MaxP_Rook, KMeans_Spatial, etc. - set this to match your clustering method
n_clusters_target = '30'  # change as needed

# Load your region metadata and interconnections
path = f'C:/Users/Benhd/OneDrive/Documents/Oxford/Year 4/4YP/Code/Detecco-Ukraine/data_prep/Clustering_Output/KNN_Graph_output/nodal_ammonia_bidirectional/{cluster_method}/{cluster_method}_k{n_clusters_target}/'
metadata_path = path + "cluster_metadata.csv"
connections_path = path + "reduced_links.csv"

cluster_df = pd.read_csv(metadata_path)
interconnections_df = pd.read_csv(connections_path)

# Create a location string from lat/lon for compatibility
cluster_df['location'] = cluster_df['lat'].round(2).astype(str) + '_' + cluster_df['lon'].round(2).astype(str)

print(f"Loaded {len(cluster_df)} clusters with demand data")
print(f"Loaded {len(interconnections_df)} interconnections")

# Build demand dictionary with CONSTANT ammonia and timeseries electricity
demand_locations_dict = {}
for idx, row in cluster_df.iterrows():
    cluster_id = int(row['cluster_id'])
    cluster_column = f'cluster_{cluster_id}'
    location_str = row['location']
    
    # Ammonia: constant value (negative for demand))
    constant_ammonia = -1 * row['ammonia_demand_MWh_hourly'] # MWh constant demand
    
    demand_locations_dict[location_str] = {
        'demand_power': f'file=cluster_demand_profiles.csv:{cluster_column}',
        'demand_ammonia': constant_ammonia  # Direct constant value, not a file reference!
    }

print(f"\n✓ Created demand dictionary")
print(f"  - Electricity demand: time-varying from cluster_demand_profiles.csv")
print(f"  - Ammonia demand: constant values (no file needed)")

Loaded 30 clusters with demand data
Loaded 66 interconnections

✓ Created demand dictionary
  - Electricity demand: time-varying from cluster_demand_profiles.csv
  - Ammonia demand: constant values (no file needed)


In [5]:
# Custom YAML representer to handle None values
class NoAliasDumper(yaml.SafeDumper):
    def ignore_aliases(self, data):
        return True

def format_region_name(lat_lon):
    return f"region_{lat_lon}".replace('.', '_')

def generate_locations_from_metadata(metadata_df, interconnections_df, demand_locations_dict):
    """
    Generate locations, links, and constraints from region metadata and interconnections CSVs.
    
    Parameters:
    - metadata_df: DataFrame with columns: region_id, lat, lon, available_area_km2, etc.
    - interconnections_df: DataFrame with columns: region_from, region_to
    - demand_locations_dict: Dict mapping location strings to demand techs
    """
    locations = {}
    links = {}
    group_constraints = {}
    
    # Update filenames to point to region-based profiles
    wind_profile_file = 'cluster_wind_profiles_weighted.csv'
    solar_profile_file = 'cluster_solar_profiles_weighted.csv'
    
    print("\nCreating locations from metadata...")
    
    # Create locations for each region
    for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Creating Locations"):
        location_str = row['location']  # e.g., "50.76_24.88"
        cluster_id = int(row['cluster_id'])  # e.g., 1, 2, 3...
        cluster_column = f'cluster_{cluster_id}'  # e.g., "region_1" # changed to cluster
        
        lat = float(row['lat'])
        lon = float(row['lon'])
        land_area = float(row['available_land_area_km2']) # In km2
        cluster_name = format_region_name(location_str)
        
        # Initialize techs dict with supply and storage technologies
        # USE cluster_column instead of location_str for the resource reference
        techs = {
            'onshore_wind': {'constraints': {'resource': f'file={wind_profile_file}:{cluster_column}'}},
            'solar_single_axis': {'constraints': {'resource': f'file={solar_profile_file}:{cluster_column}'}},
            'battery': None,
            'electrolyser': None,
            'compressed_hydrogen_storage': None,
            'fuel_cell': None,
            'haber_and_air_separation': None,
            'ammonia_storage': None,
            'ammonia_ccgt': None,
            'hydrogen_ccgt': None
        }
        
        # Add demand techs if this is a demand location
        if location_str in demand_locations_dict:
            for demand_tech, demand_profile in demand_locations_dict[location_str].items():
                techs[demand_tech] = {'constraints': {'resource': demand_profile}}
        
        # Create the location entry
        locations[cluster_name] = {
            'coordinates': {'lat': lat, 'lon': lon},
            'techs': techs
        }
        
        # Create group constraint for land area
        group_constraint_name = f"combined_wind_solar_area_limit_{cluster_name}"
        group_constraints[group_constraint_name] = {
            'techs': ['onshore_wind', 'solar_single_axis'],
            'locs': [cluster_name],
            'resource_area_max': land_area
        }
    
    # Create links from interconnections
    print("\nCreating links from interconnections...")
    for idx, row in tqdm(interconnections_df.iterrows(), total=len(interconnections_df), desc="Creating Links"):
        # Get cluster IDs
        cluster_from = row['cluster_a']
        cluster_to = row['cluster_b']
        
        # Find corresponding location strings from metadata
        loc_from = metadata_df[metadata_df['cluster_id'] == cluster_from]['location'].values[0]
        loc_to = metadata_df[metadata_df['cluster_id'] == cluster_to]['location'].values[0]
        
        # Format cluster names
        cluster_from_name = format_region_name(loc_from)
        cluster_to_name = format_region_name(loc_to)
        
        link_key = f"{cluster_from_name},{cluster_to_name}"
        
        distance_km = row['mean_distance_km']  # Use mean distance in km
        
        links[link_key] = {
            'techs': {
                'dc_transmission': {
                    'distance': distance_km
                },
                'hydrogen_pipeline': {
                    'distance': distance_km
                },
                'ammonia_pipeline': {
                    'distance': distance_km
                },
            }
        }
    
    print(f"\nCreated {len(locations)} locations")
    print(f"Created {len(links)} links")
    print(f"Created {len(group_constraints)} group constraints")
    
    return locations, links, group_constraints

In [6]:

# Generate locations and links from your CSVs
locations, links, group_constraints = generate_locations_from_metadata(
    metadata_df=cluster_df,
    interconnections_df=interconnections_df,
    demand_locations_dict=demand_locations_dict
)

# Prepare final configuration dictionary
calliope_config = {
    'locations': locations,
    'links': links,
    'group_constraints': group_constraints
}

# YAML output headers and sections
yaml_string = """
##
# LOCATIONS
##
"""

yaml_string += yaml.dump(
    {'locations': locations},
    Dumper=NoAliasDumper,
    default_flow_style=False
)

yaml_string += """
##
# TRANSMISSION CAPACITIES
##
"""
yaml_string += yaml.dump(
    {'links': links},
    Dumper=NoAliasDumper,
    default_flow_style=False
)

yaml_string += """
##
# GROUP CONSTRAINTS
##
"""
yaml_string += yaml.dump(
    {'group_constraints': group_constraints},
    Dumper=NoAliasDumper,
    default_flow_style=False
)

# Output filename
output_filename = path + "locations_custom_from_metadata.yaml"

# Write the YAML file
print(f"\nWriting the YAML file to {output_filename}...")
yaml_lines = yaml_string.splitlines()
with open(output_filename, 'w') as file:
    for line in tqdm(yaml_lines, desc="Writing YAML"):
        file.write(line + '\n')

print(f"\n✓ {output_filename} has been generated successfully!")
print(f"  - {len(locations)} locations")
print(f"  - {len(links)} interconnections")
print(f"  - {len(group_constraints)} land area constraints")



Creating locations from metadata...


Creating Locations: 100%|██████████| 30/30 [00:00<00:00, 6927.39it/s]



Creating links from interconnections...


Creating Links: 100%|██████████| 66/66 [00:00<00:00, 926.14it/s]


Created 30 locations
Created 66 links
Created 30 group constraints

Writing the YAML file to C:/Users/Benhd/OneDrive/Documents/Oxford/Year 4/4YP/Code/Detecco-Ukraine/data_prep/Clustering_Output/KNN_Graph_output/nodal_ammonia_bidirectional/MaxP_Rook/MaxP_Rook_k30/locations_custom_from_metadata.yaml...



Writing YAML: 100%|██████████| 1503/1503 [00:00<?, ?it/s]


✓ C:/Users/Benhd/OneDrive/Documents/Oxford/Year 4/4YP/Code/Detecco-Ukraine/data_prep/Clustering_Output/KNN_Graph_output/nodal_ammonia_bidirectional/MaxP_Rook/MaxP_Rook_k30/locations_custom_from_metadata.yaml has been generated successfully!
  - 30 locations
  - 66 interconnections
  - 30 land area constraints
